# V2 classified-paragraph extraction: smoke and full processing

This Colab notebook consumes documents that are already split and classified by paragraph in `classification.parquet`. It does not download, decode, or parse raw RTF documents. First validate the production prompts with isolated local smoke artifacts, then optionally promote the same prompts to the resumable full-dataset cloud pipeline.

## Dependencies

Use the project environment locally. In a clean Colab GPU runtime, uncomment and run the installation command.

In [ ]:
# Colab only:
# %pip install -q "pyarrow>=24,<25" "transformers>=5.8,<6" "accelerate>=1.13,<2" "huggingface-hub>=0.34" "json-repair>=0.50,<1" google-cloud-bigquery google-cloud-storage

## Locate and import the project

Open the notebook from the cloned repository, or set `REPOSITORY_ROOT` to the repository mounted in Colab.

In [ ]:
from pathlib import Path
import sys

import pyarrow.parquet as pq
from IPython.display import display

REPOSITORY_ROOT = Path('/content/legal_doc_parser')
if not (REPOSITORY_ROOT / 'src' / 'document_split').is_dir():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / 'src' / 'document_split').is_dir():
            REPOSITORY_ROOT = candidate
            break

source_root = REPOSITORY_ROOT / 'src'
repository_root = REPOSITORY_ROOT
if not (source_root / 'document_split').is_dir():
    raise FileNotFoundError(
        'Repository not found. Set REPOSITORY_ROOT to the cloned or mounted project.'
    )
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))
print('Repository:', repository_root)

# Real-data extraction smoke test

The input is already paragraph-split and classified. Each document gets an isolated directory containing raw handler responses, normalized JSON, final Parquet, warnings, and a recursive schema-population report.

## Select classified real decisions

The expected layout is `<classification root>/<document id>/classification.parquet`. The default three decisions cover probation, imprisonment, civil claims, costs, physical evidence, confiscation, and security measures.

In [ ]:
from document_split.v2 import (
    DEFAULT_V2_EXTRACTION_SETTINGS,
    DEFAULT_V2_PART_PROCESSING_PROMPTS,
    build_sample_processing_contexts,
    run_real_data_smoke,
)

CLASSIFICATION_ROOT = (
    source_root / 'document_split' / 'v2' / 'document_text_parsing' / 'downloads'
)
SMOKE_DOCUMENT_IDS = [
    '118355359',  # probation, costs, evidence
    '118584307',  # imprisonment, civil claim
    '116798672',  # fine, confiscation, security measures
]
CLASSIFICATION_FILES = [
    CLASSIFICATION_ROOT / document_id / 'classification.parquet'
    for document_id in SMOKE_DOCUMENT_IDS
]
SMOKE_OUTPUT_ROOT = Path('/content/v2-real-data-smoke')
RUN_EXTRACTION_SMOKE = False

missing_classifications = [
    path for path in CLASSIFICATION_FILES if not path.is_file()
]
if missing_classifications:
    raise FileNotFoundError(
        'Upload the classified inputs or change CLASSIFICATION_ROOT: '
        + ', '.join(map(str, missing_classifications))
    )

for classification_path in CLASSIFICATION_FILES:
    classification_table = pq.read_table(classification_path)
    part_column = (
        'section'
        if 'section' in classification_table.column_names
        else 'part'
    )
    sections = sorted(set(classification_table[part_column].to_pylist()))
    print(
        classification_path.parent.name,
        'paragraphs=', classification_table.num_rows,
        'sections=', sections,
    )

## Load the extraction model

Select **Runtime → Change runtime type → GPU**. Optionally expose a Colab secret named `HF_TOKEN`. The resolved model commit is printed so the smoke run is reproducible.

In [ ]:
import torch
from huggingface_hub import login, model_info

from document_split.config import MODEL_ID
from document_split.runtime import load_extraction_model

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA-enabled Colab runtime is required')

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

MODEL_REVISION = model_info(MODEL_ID, token=HF_TOKEN).sha
print('Model:', MODEL_ID)
print('Revision:', MODEL_REVISION)
model_pipe, tokenizer = load_extraction_model(
    DEFAULT_V2_EXTRACTION_SETTINGS,
    MODEL_REVISION,
    HF_TOKEN,
)

## Preview routing, batches, and token counts

This prepares the exact production-prompt messages without inference. Review the planned calls before enabling the model run.

In [ ]:
import pandas as pd
from document_split.processing import count_message_tokens

smoke_preview_rows = []
for classification_path in CLASSIFICATION_FILES:
    contexts = build_sample_processing_contexts(
        document_id=classification_path.parent.name,
        justice_kind=2,
        parts_parquet_bytes=classification_path.read_bytes(),
        tokenizer=tokenizer,
        part_prompts=DEFAULT_V2_PART_PROCESSING_PROMPTS,
    )
    for processor_name, batches in contexts.items():
        for batch_number, context in enumerate(batches, start=1):
            smoke_preview_rows.append({
                'document_id': classification_path.parent.name,
                'processor': processor_name,
                'batch': batch_number,
                'target_paragraphs': len(context.target_paragraph_ids),
                'input_tokens': count_message_tokens(
                    tokenizer, context.messages
                ),
            })

smoke_preview_df = pd.DataFrame(smoke_preview_rows)
display(smoke_preview_df)
print('Planned model calls:', len(smoke_preview_df))

## Run isolated inference

After inspecting the preview, set `RUN_EXTRACTION_SMOKE = True` in the configuration cell and run this cell.

In [ ]:
if not RUN_EXTRACTION_SMOKE:
    raise RuntimeError(
        'Review the preview, then set RUN_EXTRACTION_SMOKE = True'
    )

smoke_results = run_real_data_smoke(
    classification_files=CLASSIFICATION_FILES,
    output_root=SMOKE_OUTPUT_ROOT,
    model_pipe=model_pipe,
    tokenizer=tokenizer,
)
print('Smoke artifacts:', SMOKE_OUTPUT_ROOT)

## Review population and expected anchors

Population indicates which schema paths received non-empty values. The source-specific checks catch obvious omissions, but passing them does not replace manual comparison with the decision text.

In [ ]:
import json

population_rows = []
smoke_payloads = {}
for smoke_result in smoke_results:
    for schema_path, statistics in smoke_result.population.items():
        population_rows.append({
            'document_id': smoke_result.document_id,
            'path': schema_path,
            **statistics,
        })
    smoke_payloads[smoke_result.document_id] = json.loads(
        (smoke_result.artifact_dir / 'result.json').read_text(
            encoding='utf-8'
        )
    )

population_df = pd.DataFrame(population_rows)
display(
    population_df[population_df['populated']].sort_values(
        ['document_id', 'path']
    )
)

def nested_value(value, dotted_path):
    for key in dotted_path.split('.'):
        if not isinstance(value, dict):
            return None
        value = value.get(key)
    return value

def contains_article(value, article):
    if isinstance(value, dict):
        if str(value.get('article')) == article:
            return True
        return any(
            contains_article(child, article) for child in value.values()
        )
    if isinstance(value, list):
        return any(contains_article(child, article) for child in value)
    return False

EXPECTED_ANCHORS = {
    '118355359': {
        'article': '286',
        'paths': [
            'operative_part.final_sentence',
            'operative_part.probation',
            'operative_part.costs_reimbursement_decision',
            'operative_part.physical_evidence_decision',
        ],
    },
    '118584307': {
        'article': '185',
        'paths': [
            'operative_part.final_sentence',
            'operative_part.sentence_start',
            'operative_part.civil_claim_decision',
        ],
    },
    '116798672': {
        'article': '369-2',
        'paths': [
            'operative_part.final_sentence',
            'operative_part.security_measures_decision',
            'operative_part.physical_evidence_decision',
        ],
    },
}

anchor_rows = []
for smoke_document_id, expected in EXPECTED_ANCHORS.items():
    if smoke_document_id not in smoke_payloads:
        continue
    payload = smoke_payloads[smoke_document_id]
    anchor_rows.append({
        'document_id': smoke_document_id,
        'expectation': f"article {expected['article']}",
        'passed': contains_article(payload, expected['article']),
    })
    for expected_path in expected['paths']:
        extracted_value = nested_value(payload, expected_path)
        anchor_rows.append({
            'document_id': smoke_document_id,
            'expectation': expected_path,
            'passed': extracted_value not in (None, '', []),
        })

anchor_df = pd.DataFrame(anchor_rows)
display(anchor_df)
print('Anchor checks passed:', bool(anchor_df['passed'].all()))

for smoke_document_id, payload in smoke_payloads.items():
    print('\n===', smoke_document_id, '===')
    print(json.dumps(payload, ensure_ascii=False, indent=2)[:12000])

## Download smoke artifacts

Download the ZIP before the Colab runtime is recycled.

In [ ]:
import shutil

smoke_archive = shutil.make_archive(
    '/content/v2-real-data-smoke',
    'zip',
    SMOKE_OUTPUT_ROOT,
)
try:
    from google.colab import files
    files.download(smoke_archive)
except ImportError:
    print('Archive:', smoke_archive)

# Promote validated prompts to the full dataset

Run this section only after the smoke outputs and anchor checks are satisfactory. It uses the existing production V2 pipeline, reading classified paragraph Parquet files from the configured parts bucket and prefix. Results are written under an immutable `info_version_*` destination prefix.

The production runner records a manifest containing prompt, schema, model-revision, and batching hashes. It resumes by skipping completed `result.parquet` objects. If an existing manifest differs, the run stops and requires a new info version instead of mixing incompatible results.

## Configure the production run

Use a new `FULL_INFO_VERSION` whenever prompts, schema, model, or batching settings change. `FULL_LIMIT = None` processes the complete eligible dataset; use a small integer for a cloud staging run.

In [ ]:
from dataclasses import replace

from document_split.v2 import (
    DEFAULT_V2_CHAIN_SETTINGS,
    DEFAULT_V2_STORAGE_SETTINGS,
    run_v2_pipeline,
)

# Choose a new immutable output namespace for this prompt/schema version.
FULL_INFO_VERSION = 'info_version_13'
FULL_JUSTICE_KINDS = (2,)
FULL_LIMIT = None  # None means the complete eligible dataset.
FULL_SKIP_EXISTING = True

FULL_STORAGE_SETTINGS = replace(
    DEFAULT_V2_STORAGE_SETTINGS,
    info_version=FULL_INFO_VERSION,
    justice_kinds=FULL_JUSTICE_KINDS,
    limit=FULL_LIMIT,
    skip_existing=FULL_SKIP_EXISTING,
)
FULL_CHAIN_SETTINGS = DEFAULT_V2_CHAIN_SETTINGS

# Both safeguards must be changed deliberately.
RUN_FULL_DATASET = False
FULL_RUN_CONFIRMATION = ''  # Set exactly to: RUN FULL DATASET

print('Input bucket:', FULL_STORAGE_SETTINGS.parts_bucket)
print('Input prefix:', FULL_STORAGE_SETTINGS.parts_prefix)
print('Output bucket:', FULL_STORAGE_SETTINGS.destination_bucket)
print('Output version:', FULL_STORAGE_SETTINGS.version_prefix)
print('Justice kinds:', FULL_STORAGE_SETTINGS.justice_kinds)
print('Limit:', FULL_STORAGE_SETTINGS.limit)
print('Skip existing:', FULL_STORAGE_SETTINGS.skip_existing)

## Start or resume full processing

The pipeline reads the Colab secrets `cloud_access` and, when needed, `HF_TOKEN`. The smoke model is released first so the production runner can load the manifest-pinned model without duplicating GPU memory.

In [ ]:
if not RUN_FULL_DATASET:
    raise RuntimeError('Set RUN_FULL_DATASET = True after validation')
if FULL_RUN_CONFIRMATION != 'RUN FULL DATASET':
    raise RuntimeError(
        "Set FULL_RUN_CONFIRMATION exactly to 'RUN FULL DATASET'"
    )
if FULL_STORAGE_SETTINGS.limit is not None:
    print(
        'Staging mode: at most',
        FULL_STORAGE_SETTINGS.limit,
        'eligible documents will be considered.',
    )
else:
    print('Full mode: processing every eligible unfinished document.')

# Avoid holding two copies of the model on the GPU.
if 'model_pipe' in globals():
    del model_pipe
if 'tokenizer' in globals():
    del tokenizer
import gc
gc.collect()
torch.cuda.empty_cache()

full_run_result = run_v2_pipeline(
    extraction_settings=DEFAULT_V2_EXTRACTION_SETTINGS,
    storage_settings=FULL_STORAGE_SETTINGS,
    chain_settings=FULL_CHAIN_SETTINGS,
    part_prompts=DEFAULT_V2_PART_PROCESSING_PROMPTS,
)
print('Full-run counters:', full_run_result)